In [2]:
# ============================================================
# SCRIPT 2 of 2 — SPLIT ONLY
# Reads the recovered 76-subject pool (_train_subs_ORIGINAL_76.npy,
# produced by 1_recover_only.py) and the untouched _test_subs.npy,
# generates a fresh train/val split, and saves all three lists to
# a DIFFERENT output directory.
#
# Run 1_recover_only.py first if you haven't already.
# ============================================================

import os
import random
import numpy as np

# ============================================================
# CONFIG
# ============================================================
OLD_SAVE_PATH = r"D:\22\AA\preprocess\preprocessed_FFinal"        # where _train_subs_ORIGINAL_76.npy and _test_subs.npy live
NEW_SAVE_PATH = r"D:\22\AA\AA journal\preprocess\preprocessed_split_v2"       # <-- your new output folder

VAL_FRACTION = 0.15
SPLIT_SEED   = 42

os.makedirs(NEW_SAVE_PATH, exist_ok=True)

# ============================================================
# LOAD (read-only -- nothing in OLD_SAVE_PATH gets modified)
# ============================================================
original_pool_path = os.path.join(OLD_SAVE_PATH, "_train_subs.npy")
test_path          = os.path.join(OLD_SAVE_PATH, "_test_subs.npy")

if not os.path.exists(original_pool_path):
    raise FileNotFoundError(
        f"{original_pool_path} not found. Run 1_recover_only.py first."
    )

original_pool = np.load(original_pool_path, allow_pickle=True).tolist()
TEST_SUBS     = np.load(test_path,          allow_pickle=True).tolist()

print(f"Loaded original pool: {len(original_pool)} subjects")
print(f"Loaded test set     : {len(TEST_SUBS)} subjects")

# ============================================================
# SPLIT into train + val
# ============================================================
pool = original_pool.copy()
rng = random.Random(SPLIT_SEED)
rng.shuffle(pool)

n_val = max(1, int(round(VAL_FRACTION * len(pool))))
VAL_SUBS   = sorted(pool[:n_val])
TRAIN_SUBS = sorted(pool[n_val:])

# Safety checks
assert set(TRAIN_SUBS).isdisjoint(VAL_SUBS),  "TRAIN and VAL overlap!"
assert set(TRAIN_SUBS).isdisjoint(TEST_SUBS), "TRAIN and TEST overlap!"
assert set(VAL_SUBS).isdisjoint(TEST_SUBS),   "VAL and TEST overlap!"
assert set(TRAIN_SUBS) | set(VAL_SUBS) == set(original_pool), \
    "Some subjects got lost or duplicated!"

print(f"\nNew split created (verified, no overlap):")
print(f"  Train : {len(TRAIN_SUBS)} subjects")
print(f"  Val   : {len(VAL_SUBS)} subjects")
print(f"  Test  : {len(TEST_SUBS)} subjects")

# ============================================================
# SAVE to the new directory
# ============================================================
np.save(os.path.join(NEW_SAVE_PATH, "_train_subs.npy"), np.array(TRAIN_SUBS))
np.save(os.path.join(NEW_SAVE_PATH, "_val_subs.npy"),   np.array(VAL_SUBS))
np.save(os.path.join(NEW_SAVE_PATH, "_test_subs.npy"),  np.array(sorted(TEST_SUBS)))

print(f"\nSaved to: {NEW_SAVE_PATH}")
print(f"  _train_subs.npy ({len(TRAIN_SUBS)})")
print(f"  _val_subs.npy   ({len(VAL_SUBS)})")
print(f"  _test_subs.npy  ({len(TEST_SUBS)})")
print(f"\nPoint your training scripts' SAVE_PATH to:\n  {NEW_SAVE_PATH}")

Loaded original pool: 76 subjects
Loaded test set     : 20 subjects

New split created (verified, no overlap):
  Train : 65 subjects
  Val   : 11 subjects
  Test  : 20 subjects

Saved to: D:\22\AA\AA journal\preprocess\preprocessed_split_v2
  _train_subs.npy (65)
  _val_subs.npy   (11)
  _test_subs.npy  (20)

Point your training scripts' SAVE_PATH to:
  D:\22\AA\AA journal\preprocess\preprocessed_split_v2
